# 0. Imports

## 0.1 Packages

In [32]:
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import pandas as pd

## 0.2 Data

In [33]:
ObsList = pd.read_csv(r"../Data Raw/ObsList.csv", sep=";")
NativeData = pd.read_csv(r"../Data Raw/NativeData.csv", sep=";")

# 1. Taxonomy Uniformization

## 1.1. Code

### 1.1.1. GBIF: Names Check

#### 1.1.1.1. Uniformization

In [34]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_1(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return species
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_1(species_list))

#### 1.1.1.2. Filter

In [35]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_2(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return None
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_2(species_list))

### 1.1.2. Global Names Verifier: Cross-check

#### 1.1.2.1. Uniformization

In [36]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_1(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name if accepted_name != "" else species
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_1(species_list))

## 1.2. Data

### 1.2.1. Observation Data

In [40]:
ObsList['Species'] = ObsList['Species'].apply(lambda x: ' '.join(x.split()[:2]))

In [41]:
ObsList['AcceptedSpecies'] = GBIF_Extractor_2(VNF_Extractor_1(GBIF_Extractor_1(ObsList['Species'])))
ObsList['AcceptedSpecies'] = ObsList['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

In [42]:
ObsList[ObsList['AcceptedSpecies'].isna()]['Species'].unique()

array(['Heliocheilus cystiphora', 'Rheumaptera affirmata',
       'Spodoptera sunia', 'Heliocontia margana', 'Trissodoris guamensis',
       'Prospalta dolorosa', 'Phalaenophana fadusalis',
       'Opodiphthera eucalypti', 'Euphaedra temeraria', 'Orgyia basalis'],
      dtype=object)

In [43]:
ObsList[ObsList['AcceptedSpecies'].isna()].shape[0]


13

In [44]:
ObservationsClean = ObsList.copy()
ObservationsClean.dropna(subset=['AcceptedSpecies'], inplace=True)
ObservationsClean[ObservationsClean['AcceptedSpecies'].isna()]

,Species,NAME_0,Realm,Cryptogenic,Dispersal,Eradicated,IntentionalRelease,Introduced,Established,ReportedFirstYear,Reference,ReferenceYear,AcceptedSpecies


In [45]:
ObservationsClean.reset_index(drop=True, inplace=True)
ObservationsClean.to_csv(r'../Transformed Data/ObservationsClean.csv', sep =';')

### 1.2.2. Native Distribution Data

In [46]:
NativeData['Species'] = NativeData['Species'].apply(lambda x: ' '.join(x.split()[:2]))

In [47]:
NativeData['AcceptedSpecies'] = GBIF_Extractor_2(VNF_Extractor_1(GBIF_Extractor_1(NativeData['Species'])))
NativeData['AcceptedSpecies'] = NativeData['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

In [48]:
NativeData[NativeData['AcceptedSpecies'].isna()]['Species'].unique()

array([], dtype=object)

In [49]:
NativeDataClean = NativeData[NativeData['AcceptedSpecies'].isin(ObservationsClean['AcceptedSpecies'])].copy()

In [50]:
NativeDataClean.drop(columns=['Unnamed: 0'], inplace=True)
NativeDataClean.reset_index(drop=True, inplace=True)
NativeDataClean.to_csv(r'../Transformed Data/NativeDataClean.csv', sep =';')